# 05. Codeforces data collection + 3-way classification (RQ2)

Codeforces API to collect rated users and classify profile pictures with yolov8_animeface.

**Output**: `data/processed/codeforces_classified.csv`

In [ ]:
TOTAL_USERS = 1000

In [ ]:
import json
import sys
import asyncio
from pathlib import Path

import aiohttp
import pandas as pd
import requests
from tqdm import tqdm

PROJECT_DIR = Path('../').resolve()
DATA_DIR = PROJECT_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
PROC_DIR = DATA_DIR / 'processed'
CF_AVATAR_DIR = RAW_DIR / 'avatars_cf'
CF_AVATAR_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from device import get_device
DEVICE = get_device()
print(f'Project: {PROJECT_DIR}')
print(f'Device: {DEVICE}')

## 1. Codeforces Rated user collection

In [ ]:
CF_USERS_PATH = RAW_DIR / 'codeforces_users.json'

if not CF_USERS_PATH.exists():
    print('Codeforces rated user collecting...')
    resp = requests.get('https://codeforces.com/api/user.ratedList?activeOnly=true')
    resp.raise_for_status()
    data = resp.json()
    users = data['result']
    with open(CF_USERS_PATH, 'w', encoding='utf-8') as f:
        json.dump(users, f, ensure_ascii=False)
    print(f'Collection done: {len(users)} users')
else:
    with open(CF_USERS_PATH, encoding='utf-8') as f:
        users = json.load(f)
    print(f'Loaded existing data: {len(users)} users')

cf_df = pd.DataFrame(users)
print(f'Rank distribution:\n{cf_df["rank"].value_counts()}')

In [ ]:
import numpy as np

N_BINS = 10
SEED = 42

cf_df['rating_bin'] = pd.qcut(cf_df['rating'], q=N_BINS, duplicates='drop')
actual_bins = cf_df['rating_bin'].cat.categories.size
base = TOTAL_USERS // actual_bins
remainder = TOTAL_USERS - base * actual_bins

rng = np.random.default_rng(SEED)
sampled_idx = []
bin_counts = []
for i, (bin_label, g) in enumerate(cf_df.groupby('rating_bin', observed=True)):
    quota = base + (1 if i < remainder else 0)
    n = min(quota, len(g))
    bin_counts.append((str(bin_label), n, len(g)))
    sampled_idx.extend(rng.choice(g.index, size=n, replace=False))

cf_df = cf_df.loc[sampled_idx].drop(columns='rating_bin').reset_index(drop=True)
print(f'Stratified sample by rating: {len(cf_df)}/{TOTAL_USERS} users '
      f'({actual_bins} bins, seed={SEED})')
for label, taken, avail in bin_counts:
    print(f'  {label}: {taken}/{avail}')
print('rating:', cf_df['rating'].describe().round(1).to_dict())
print('rank:\n', cf_df['rank'].value_counts())

## 2. Default Avatar Filtering + Image Download

In [ ]:
def get_cf_avatar_url(row):
    url = row.get('titlePhoto', row.get('avatar', ''))
    if url and not url.startswith('http'):
        url = 'https:' + url
    return url

cf_df['avatar_url'] = cf_df.apply(get_cf_avatar_url, axis=1)

DEFAULT_PATTERNS = ['/no-avatar.jpg', '/no-title.jpg', 'userpic.codeforces.org/no']
def is_cf_default(url):
    if not url:
        return True
    return any(p in url for p in DEFAULT_PATTERNS)

cf_df['is_default'] = cf_df['avatar_url'].apply(is_cf_default)
print(f'Default avatars: {cf_df["is_default"].sum()} users')
print(f'Download targets: {(~cf_df["is_default"]).sum()} users')

In [ ]:
to_download = cf_df[~cf_df['is_default']].copy()

async def download_cf_avatars(df, output_dir, concurrency=30):
    semaphore = asyncio.Semaphore(concurrency)
    failed = []
    existing = {p.stem for p in output_dir.glob('*.png')}
    targets = [(row['handle'], row['avatar_url'])
               for _, row in df.iterrows()
               if row['handle'] not in existing]
    print(f'Download: {len(targets)} (existing: {len(existing)})')
    if not targets:
        return
    async with aiohttp.ClientSession() as session:
        pbar = tqdm(total=len(targets), desc='CF avatars')
        async def _dl(handle, url):
            async with semaphore:
                try:
                    async with session.get(url, timeout=aiohttp.ClientTimeout(total=30)) as resp:
                        if resp.status == 200:
                            data = await resp.read()
                            (output_dir / f'{handle}.png').write_bytes(data)
                        else:
                            failed.append((handle, resp.status))
                except Exception as e:
                    failed.append((handle, str(e)))
                finally:
                    pbar.update(1)
        await asyncio.gather(*[_dl(h, u) for h, u in targets])
        pbar.close()
    if failed:
        print(f'Failed: {len(failed)}')
    print(f'Done. Total images: {len(list(output_dir.glob("*.png")))}')

await download_cf_avatars(to_download, CF_AVATAR_DIR)

In [ ]:
from ultralytics import YOLO

MODEL_PATH = PROJECT_DIR / 'yolov8x6_animeface.pt'
assert MODEL_PATH.exists(), f'Model not found: {MODEL_PATH}'
model = YOLO(str(MODEL_PATH))
CONF_THRESHOLD = 0.3

cf_df['profile_type'] = 'pending'
cf_df.loc[cf_df['is_default'], 'profile_type'] = 'Default'

remaining = cf_df[cf_df['profile_type'] == 'pending']
print(f'yolov8 Detection targets: {len(remaining)} users (device={DEVICE})')

anime_results = []
for _, row in tqdm(remaining.iterrows(), total=len(remaining), desc='Anime detection'):
    img_path = CF_AVATAR_DIR / f"{row['handle']}.png"
    if not img_path.exists():
        anime_results.append((row['handle'], False, 0.0, 0))
        continue
    try:
        results = model.predict(str(img_path), conf=CONF_THRESHOLD, iou=0.5, verbose=False, device=DEVICE)
        boxes = results[0].boxes
        if len(boxes) > 0:
            anime_results.append((row['handle'], True, float(boxes.conf.max()), len(boxes)))
        else:
            anime_results.append((row['handle'], False, 0.0, 0))
    except:
        anime_results.append((row['handle'], False, 0.0, 0))

anime_res_df = pd.DataFrame(anime_results, columns=['handle', 'is_anime', 'anime_conf', 'anime_faces'])
cf_df = cf_df.merge(anime_res_df, on='handle', how='left')

cf_df.loc[(cf_df['profile_type'] == 'pending') & (cf_df['is_anime'] == True), 'profile_type'] = 'Anime'
cf_df.loc[cf_df['profile_type'] == 'pending', 'profile_type'] = 'Photo'

print(f'\n=== Codeforces 3-Way Classification Results ===')
print(cf_df['profile_type'].value_counts())

In [ ]:
keep_cols = ['handle', 'rating', 'maxRating', 'rank', 'maxRank',
             'avatar_url', 'profile_type', 'is_anime', 'anime_conf', 'anime_faces',
             'country', 'organization', 'registrationTimeSeconds']
keep_cols = [c for c in keep_cols if c in cf_df.columns]

output_path = PROC_DIR / 'codeforces_classified.csv'
if not PROC_DIR.exists():
    PROC_DIR.mkdir(parents=True, exist_ok=True)
cf_df[keep_cols].to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'Saved: {output_path}')
print(f'Total {len(cf_df)} users')
cf_df[keep_cols].head()